In [1]:
import torch
import torch.nn as nn

In [2]:
class Encoder(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.conv1 = nn.Conv2d(1, d, 4, padding=1, stride=2)
        self.conv2 = nn.Conv2d(d, d, 4, padding=1, stride=2)
        self.act   = nn.ReLU()

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        return x

In [3]:
class Decoder(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.convt1 = nn.ConvTranspose2d(d, d, 4, padding=1, stride=2)
        self.convt2 = nn.ConvTranspose2d(d, 1, 4, padding=1, stride=2)
        self.act   = nn.ReLU()

    def forward(self, x):
        x = self.act(self.convt1(x))
        x = self.convt2(x)
        return x

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, K=128, d=32, gamma=0.99, eps=1e-5, dead_threshold=1.0):
        super().__init__()
        self.K              = K
        self.d              = d
        self.gamma          = gamma
        self.eps            = eps
        self.dead_threshold = dead_threshold

        self.register_buffer('codebook',         torch.randn(K, d))
        self.register_buffer('ema_cluster_size', torch.zeros(K))
        self.register_buffer('ema_embed_sum',    torch.randn(K, d))

    def forward(self, z_e):
        B, C, H, W = z_e.shape
        z_e_flat = z_e.view(B, C, -1).transpose(1, 2).contiguous()  # (B, H*W, d)

        dists = torch.cdist(z_e_flat, self.codebook)     # (B, H*W, K)
        idx   = torch.argmin(dists, dim=2)               # (B, H*W)

        if self.training:
            with torch.no_grad():
                one_hot = torch.zeros(idx.numel(), self.K, device=z_e.device)
                one_hot.scatter_(1, idx.view(-1, 1), 1)

                n_k = one_hot.sum(0)
                self.ema_cluster_size = self.gamma * self.ema_cluster_size \
                                      + (1 - self.gamma) * n_k

                dw = one_hot.t() @ z_e_flat.view(-1, self.d)
                self.ema_embed_sum = self.gamma * self.ema_embed_sum \
                                   + (1 - self.gamma) * dw

                n        = self.ema_cluster_size.sum()
                n_smooth = (self.ema_cluster_size + self.eps) \
                         / (n + self.K * self.eps) * n
                self.codebook = self.ema_embed_sum / n_smooth.view(self.K, 1)

                dead   = self.ema_cluster_size < self.dead_threshold
                n_dead = dead.sum().item()
                if n_dead > 0:
                    perm = torch.randperm(z_e_flat.shape[0] * z_e_flat.shape[1], device=z_e.device)[:n_dead]
                    self.codebook[dead] = z_e_flat.view(-1, self.d)[perm]

        z_q = self.codebook[idx]                          # (B, H*W, d)
        z_q = z_q.transpose(1, 2).contiguous().view(B, C, H, W)

        z_q_st = z_e + (z_q - z_e).detach()
        return z_q_st, z_q, idx.view(B, H, W)

In [ ]:
class VQVAE(nn.Module):
    def __init__(self, K=128, d=32):
        super().__init__()
        self.enc = Encoder(d)
        self.vq  = VectorQuantizer(K, d)
        self.dec = Decoder(d)

    def forward(self, x):
        z_e              = self.enc(x)
        z_q_st, z_q, idx = self.vq(z_e)
        x_hat            = self.dec(z_q_st)
        return x_hat, z_e, z_q, idx

In [6]:
model = VQVAE()
x = torch.randn(4, 1, 28, 28)
print(f'input:  {x.shape}')   # expect (4, 1, 28, 28)
x_hat, z_e, z_q, idx = model(x)
print(f'x_hat: {x_hat.shape}') # expect (4, 1, 28, 28)
print(f'z_e: {z_e.shape}') # expect (4, d, 7, 7)
print(f'z_q: {z_q.shape}') # expect (4, d, 7, 7)
print(f'idx: {idx.shape}') # expect (4, 7, 7)

input:  torch.Size([4, 1, 28, 28])
x_hat: torch.Size([4, 1, 28, 28])
z_e: torch.Size([4, 32, 7, 7])
z_q: torch.Size([4, 32, 7, 7])
idx: torch.Size([4, 7, 7])
